<!-- nav -->
[← 8.09 · Cpp Types](08_09_Cpp_Types.ipynb) · [Home](../../README.md) · [8.11 · POSIX BRE →](08_11_POSIX_BRE.ipynb)

# 8.10 · Cpp RegExp

C++'s library reads regular expressions: `std::regex r("ab|c");` compiles its string argument as a pattern of
[re.grammar]'s grammar, ECMAScript's (ECMA-262, its third edition), as [re.grammar] modifies it. To C++ the pattern is
a string, a literal like any other: no C++ subterm is a regular expression, and none is guessed to be one
([PRI-025: never guess the schema](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-025)). The
pattern's own grammar is a dialect of its own, `Cpp RegExp`, whose holder holds one. Its kinds are ECMAScript's, named as
regexpp, ECMAScript's reference parser, names them. And a `regex` names its grammar among its flags: five of
[re.synopt]'s flags select a grammar POSIX defines, whose pattern is a term of POSIX's own dialects,
[8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb) and [8.12 · POSIX ERE](08_12_POSIX_ERE.ipynb).

This page builds a pattern from two characters up, then ECMAScript's constructs, then a `regex` and its flags, one
whose flags name a POSIX grammar among them; then what validation says, and the editions it covers. It starts from
`dialects()`; `S` is its holder namespace. `char` writes a character, of its code point, as a spec the builders call,
and `alternative` and `pattern` the two kinds every pattern is made of, once 08.10.1 has written them in full:

In [1]:
import assert from "node:assert/strict";
import * as Cpp from "@mbse/expressions/Cpp/Expressions";
import * as CppRegExp from "@mbse/expressions/Cpp/RegExp";
import { Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const char = (c: string) => (u: any) => u.character((r: any) => r.value(BigInt(c.codePointAt(0)!)));
const alternative = (...elements: unknown[]) => (u: any) => u.alternative((r: any) => r.elements(elements));
const pattern = (...alternatives: unknown[]) => (u: any) => u.pattern((r: any) => r.alternatives(alternatives));
const XR = CppRegExp.Expressions;
const DIALECT = XR.DIALECT;

/** What `simplify` says of `expression`: its value, a simplification's where its residual is a literal, or why it has
 * none (`NotEvaluated`). */
function refused(simplify: (expression: any) => unknown, expression: any): string {
  try {
    const result: any = simplify(expression);
    return String(typeof result?.evaluate === "function" ? result.evaluate() : result);
  } catch (error) {
    return (error as Error).message;
  }
}

<a id="T08.10.1"></a>
## 08.10.1 · A pattern, `ab|c`

```text
ab|c
```

A pattern is a disjunction of alternatives, each a sequence of terms (ECMA-262's *Disjunction*, *Alternative*, *Term*):
`ab|c` is a `pattern` whose `alternatives` are two `alternative`s, the first of the characters `a` and `b`, the second
of `c`. A `character` holds its code point, `value`, as regexpp does, and its spelling where the text gives it one of
its own (`\x41`). `S.Cpp.RegExp.Expression` holds it; `CppRegExp.Text.ToText` prints it as ECMAScript writes it:

In [2]:
const abc = store.transaction(() => S.Cpp.RegExp.Expression((e: any) => e.root((u: any) => u.pattern((p: any) => p.alternatives([
  (a: any) => a.alternative((r: any) => r.elements([
    (c: any) => c.character((r: any) => r.value(BigInt("a".codePointAt(0)!))), (c: any) => c.character((r: any) => r.value(BigInt("b".codePointAt(0)!)))])),
  (a: any) => a.alternative((r: any) => r.elements([(c: any) => c.character((r: any) => r.value(BigInt("c".codePointAt(0)!)))]))])))));
console.log(CppRegExp.Text.ToText(abc), "|", Terms.kindOf(Terms.term(abc))[0], "of", Terms.dialectOf(Terms.term(abc)));
for (const t of Terms.walk(abc)) {
  console.log(" ", Terms.kindOf(t)[0], ...[...Terms.form(t).attributes].map(([k, v]) => `${k}=${v}`));
}
console.log(DIALECT.name(), DIALECT.tag, "| its kinds:", [...DIALECT.kinds().keys()].join(", "));
assert.equal(CppRegExp.Text.ToText(abc), "ab|c");

ab|c | pattern of Expressions.Cpp.RegExp.Term


  pattern


  alternative


  character value=97


  character value=98


  alternative


  character value=99


Cpp RegExp cpp_regexp | its kinds: regex, pattern, alternative, group, capturinggroup, quantifier, characterclass, characterclassrange, characterset, character, assertion, backreference, exclass, collatingelement, equivalence, error, missing


<a id="T08.10.2"></a>
## 08.10.2 · ECMAScript's constructs

```text
^(a|\.)b*c+\d{2,3}[^a-z]$        (?:a)(?=b)x*?\b
```

Each construct of ECMAScript's grammar is a kind: an `assertion`, `^` and `$` (its `assertionKind`, `start` or `end`;
`\b`, `word`; `(?=b)`, a `lookahead` holding alternatives); a `capturinggroup` and a non-capturing `group`; a
`quantifier`, its `min` and `max` and whether it is `greedy`, repeating one `element`; a `characterset`, `\d` or `.`
(its `setKind`); a `characterclass`, `[^a-z]`, holding characters and `characterclassrange`s. The printer escapes what
ECMAScript reads as syntax, so the character `.` prints `\.`:

In [3]:
const repeat = (element: unknown, least: bigint, most?: bigint) => (u: any) => u.quantifier(
  (r: any) => most === undefined ? r.min(least).element(element) : r.min(least).max(most).element(element));
const [constructs, ecmascript] = store.transaction(() => [
  S.Cpp.RegExp.Expression((e: any) => e.root(pattern(alternative(
    (u: any) => u.assertion((r: any) => r.assertionKind("start")),
    (u: any) => u.capturinggroup((r: any) => r.alternatives([alternative(char("a")), alternative(char("."))])),
    repeat(char("b"), 0n), repeat(char("c"), 1n),
    repeat((u: any) => u.characterset((r: any) => r.setKind("digit")), 2n, 3n),
    (u: any) => u.characterclass((r: any) => r.negate(true).elements([
      (u: any) => u.characterclassrange((r: any) => r.min(char("a")).max(char("z")))])),
    (u: any) => u.assertion((r: any) => r.assertionKind("end")))))),
  S.Cpp.RegExp.Expression((e: any) => e.root(pattern(alternative(
    (u: any) => u.group((r: any) => r.alternatives([alternative(char("a"))])),
    (u: any) => u.assertion((r: any) => r.assertionKind("lookahead").alternatives([alternative(char("b"))])),
    (u: any) => u.quantifier((r: any) => r.min(0n).greedy(false).element(char("x"))),
    (u: any) => u.assertion((r: any) => r.assertionKind("word")))))),
]);
for (const expression of [constructs, ecmascript]) {
  console.log(CppRegExp.Text.ToText(expression));
  console.log(" ", [...new Set([...Terms.walk(expression)].map((t) => Terms.kindOf(t)[0]))].join(", "));
}
assert.equal(CppRegExp.Text.ToText(constructs), "^(a|\\.)b*c+\\d{2,3}[^a-z]$");
assert.equal(CppRegExp.Text.ToText(ecmascript), "(?:a)(?=b)x*?\\b");

^(a|\.)b*c+\d{2,3}[^a-z]$


  pattern, alternative, assertion, capturinggroup, character, quantifier, characterset, characterclass, characterclassrange


(?:a)(?=b)x*?\b


  pattern, alternative, group, character, assertion, quantifier


<a id="T08.10.3"></a>
## 08.10.3 · A `regex` and its flags

```text
std::regex r("ab|c", std::regex::icase);
```

A `std::regex` is built of its pattern and its `syntax_option_type` flags. The dialect's `regex` kind holds both: its
`flags` as written, and its `pattern`. The flags name at most one grammar; none names ECMAScript's, the default, whose
pattern is a `Cpp RegExp` term. The printer writes the pattern alone, since the flags are the constructor's other
argument. Which dialect each grammar's pattern is of is data the dialect holds, `GRAMMAR_DIALECTS`:

In [4]:
const insensitive = store.transaction(() => S.Cpp.RegExp.Expression((e: any) => e.root((u: any) => u.regex((r: any) => r.flags(["icase"])
  .pattern((t: any) => t.cpp_regexp(pattern(alternative(char("a"), char("b")), alternative(char("c")))))))));
const form = Terms.form(Terms.term(insensitive));
const flags = (form.attributes.get("flags") as any).values as string[];
console.log(CppRegExp.Text.ToText(insensitive), "| flags:", flags.join(", "),
  "| its grammar:", XR.grammar(flags), "| its pattern:", Terms.dialectOf(form.arguments[0]));
for (const [grammar, dialect] of XR.GRAMMAR_DIALECTS) {
  console.log(`  ${grammar.padEnd(10)}`, dialect);
}
assert.ok(DIALECT.validate(insensitive).length === 0 && CppRegExp.Text.ToText(insensitive) === "ab|c");

ab|c | flags: icase | its grammar: ECMAScript | its pattern: Expressions.Cpp.RegExp.Term


  ECMAScript Cpp.RegExp


  basic      POSIX.BRE


  extended   POSIX.ERE


  awk        POSIX.ERE


  grep       POSIX.BRE


  egrep      POSIX.ERE


<a id="T08.10.4"></a>
## 08.10.4 · A POSIX grammar's pattern is POSIX's

```text
std::regex r("^(a)b+\n$", std::regex::extended);        std::regex r("^\\(a\\)b\\{1,\\}\n$", std::regex::basic);
```

[re.synopt]'s `basic` and `grep` select POSIX's basic regular expressions, and `extended`, `awk` and `egrep` its
extended ones, grammars [re.grammar] does not define, but POSIX does (IEEE Std 1003.1, XBD 9). So a `regex` whose flags
name one holds a pattern of POSIX's dialect, `POSIX BRE` or `POSIX ERE`, as a C++ term is read under the ABI its holder
names, and `Cpp RegExp` reads it as that dialect reads it. The same tree, a start anchor, a group of `a`, `b` repeated
at least once, a newline, an end anchor, is a POSIX ERE term under `extended`, printed as an ERE writes it, a POSIX BRE
term under `basic`, printed as a BRE writes it, with its backslashes, and an ERE under `awk`, which writes a newline as C
does, `\n`, where an ERE writes the newline itself (`⏎` below):

In [5]:
const anchored = pattern(alternative(
  (u: any) => u.assertion((r: any) => r.assertionKind("start")),
  (u: any) => u.capturinggroup((r: any) => r.alternatives([alternative(char("a"))])),
  (u: any) => u.quantifier((r: any) => r.min(1n).element(char("b"))), char("\n"),
  (u: any) => u.assertion((r: any) => r.assertionKind("end"))));
const posix: Record<string, any> = store.transaction(() => Object.fromEntries([
  ["extended", "posix_ere"], ["basic", "posix_bre"], ["awk", "posix_ere"],
].map(([flag, branch]) => [flag, S.Cpp.RegExp.Expression((e: any) => e.root((u: any) => u.regex(
  (r: any) => r.flags([flag]).pattern((t: any) => t[branch!](anchored)))))])));
for (const [flag, regex] of Object.entries(posix)) {
  const held = Terms.form(Terms.term(regex)).arguments[0];
  console.log(flag.padEnd(8), CppRegExp.Text.ToText(regex).replaceAll("\n", "⏎").padEnd(18), "| its pattern:", DIALECT.owner(held).name(),
    "|", DIALECT.validate(regex).join("; ") || "valid");
}
assert.deepEqual(Object.values(posix).map((r) => CppRegExp.Text.ToText(r)), ["^(a)b+\n$", "^\\(a\\)b\\{1,\\}\n$", "^(a)b+\\n$"]);

extended ^(a)b+⏎$           | its pattern: POSIX ERE | valid


basic    ^\(a\)b\{1,\}⏎$    | its pattern: POSIX BRE | valid


awk      ^(a)b+\n$          | its pattern: POSIX ERE | valid


What C++ selects of POSIX is checked under the regex. A grammar's pattern is a term of its dialect; and what POSIX.1-2024
added, which the editions C++ cites lack, is refused under C++'s grammars, though POSIX's dialects hold it: a BRE's
alternation, `\|`, and grep's list of patterns, which only `grep` and `egrep` read:

In [6]:
const wrong: Record<string, any> = store.transaction(() => Object.fromEntries(([
  ["ERE's term under basic", ["basic"], (t: any) => t.posix_ere(pattern(alternative(char("a"))))],
  ["ab\\|c under basic", ["basic"], (t: any) => t.posix_bre(pattern(alternative(char("a"), char("b")), alternative(char("c"))))],
  ["a list under awk", ["awk"], (t: any) => t.posix_ere((u: any) => u.patternlist((r: any) => r.patterns([
    pattern(alternative(char("a"))), pattern(alternative(char("b")))])))],
  ["basic and extended", ["basic", "extended"], (t: any) => t.posix_bre(pattern(alternative(char("a"))))],
] as [string, string[], unknown][]).map(([source, flags, spec]) => [source, S.Cpp.RegExp.Expression((e: any) => e.root((u: any) => u.regex(
  (r: any) => r.flags(flags).pattern(spec))))])));
for (const [source, regex] of Object.entries(wrong)) {
  console.log(source.padEnd(20), "|", DIALECT.validate(regex).join("; "));
}
assert.deepEqual(DIALECT.validate(wrong["ERE's term under basic"]), [
  "the basic grammar's pattern is a POSIX BRE term, got a POSIX ERE term"]);

ERE's term under basic | the basic grammar's pattern is a POSIX BRE term, got a POSIX ERE term


ab\|c under basic    | alternation is not in the basic grammar


a list under awk     | a patternlist is not in the awk grammar


basic and extended   | a regex has at most one grammar element, got basic, extended


<a id="T08.10.5"></a>
## 08.10.5 · Checked, binding nothing, not simplified

Validation checks what each kind's terms may be, as ECMAScript's grammar has them: a quantifier repeats an atom, so a
quantifier of a quantifier, `a+*`, is refused, as is a quantifier's `max` below its `min`. A capturing group binds no
name a term scopes: groups are numbered, and a backreference, `\1`, names one by its number, so a pattern needs nothing
from its scope. No rule simplifies a regular expression: C++'s rules leave one in the residual, named by its dialect:

In [7]:
const checked: Record<string, any> = store.transaction(() => ({
  "a+*": S.Cpp.RegExp.Expression((e: any) => e.root(pattern(alternative(repeat(repeat(char("a"), 1n), 0n))))),
  "a{3,2}": S.Cpp.RegExp.Expression((e: any) => e.root(pattern(alternative(repeat(char("a"), 3n, 2n))))),
  "(a)\\1": S.Cpp.RegExp.Expression((e: any) => e.root(pattern(alternative(
    (u: any) => u.capturinggroup((r: any) => r.alternatives([alternative(char("a"))])),
    (u: any) => u.backreference((r: any) => r.ref(1n)))))),
}));
for (const [source, expression] of Object.entries(checked)) {
  console.log(source.padEnd(7), "|", DIALECT.validate(expression).join("; ") || "valid",
    "| needs:", [...Symbolics.free(expression, DIALECT)].sort().join(", ") || "nothing");
}
console.log(refused((x) => Cpp.Simplifiers.OfAny(x).evaluate(), insensitive));
assert.equal(CppRegExp.Text.ToText(checked["(a)\\1"]), "(a)\\1");

a+*     | argument 0: argument 0: a quantifier's element cannot hold a quantifier | needs: nothing


a{3,2}  | argument 0: argument 0: a quantifier's max cannot be less than its min, got 2 and 3 | needs: nothing


(a)\1   | valid | needs: nothing


the root is not a literal: Cpp RegExp regex is not simplified


<a id="T08.10.6"></a>
## 08.10.6 · Its editions

`Cpp RegExp` covers C++'s three editions, each by its designation and its common name, every one of its kinds in each,
since [re.grammar] has not changed since C++11 ([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)).
The editions of the POSIX grammars a regex selects are POSIX's own, which their pages read:

In [8]:
const regexp = store.Instances.Expressions.Cpp.RegExp; // the dialect, by its global name
for (const [designation, edition] of Standards.standards(regexp)) {
  console.log(designation, "|", Standards.tags(edition).slice(1).join(", "), "|", Standards.kinds(edition).length, "kinds");
}
assert.ok(Standards.find(regexp, "C++17") === Standards.standards(regexp).get("ISO/IEC 14882:2017"));

ISO/IEC 14882:2017 | C++17 | 15 kinds


ISO/IEC 14882:2020 | C++20 | 15 kinds


ISO/IEC 14882:2024 | C++23 | 15 kinds


## What you learned

- **A C++ regular expression is a `Cpp RegExp` term**, held by its own holder, since to C++ a pattern is a string.
- **Its kinds are ECMAScript's**, as regexpp names them: a pattern of alternatives, assertions, groups, quantifiers,
  character sets and classes, backreferences.
- **A `regex` keeps its flags as written**, and its flags name its grammar: ECMAScript's by default, or one of POSIX's,
  whose pattern is a term of [8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb) or [8.12 · POSIX ERE](08_12_POSIX_ERE.ipynb), printed
  as POSIX writes it.
- **What C++ selects of POSIX is checked under the regex**, and nothing is simplified.

Next, POSIX's basic regular expressions, [8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb).

<!-- nav -->
[← 8.09 · Cpp Types](08_09_Cpp_Types.ipynb) · [Home](../../README.md) · [8.11 · POSIX BRE →](08_11_POSIX_BRE.ipynb)